# 📊 Fabric Racing Game - Live Dashboard

Real-time analytics for race telemetry using KQL.

**Visualizations:**
- Live positions on track
- Speed trends
- Lap times comparison
- Event heatmap

In [ ]:
# Install required packages (run this cell first!)
%pip install azure-kusto-data azure-identity plotly -q

In [ ]:
# Configuration
KUSTO_CLUSTER = "<YOUR_EVENTHOUSE_URI>"  # e.g., https://xxx.kusto.fabric.microsoft.com
DATABASE_NAME = "RaceData"

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output
import time

In [ ]:
# Initialize Kusto client with Managed Identity (Fabric notebooks)
kcsb = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER)
client = KustoClient(kcsb)

In [ ]:
def run_query(query: str):
    """Execute KQL query and return DataFrame"""
    response = client.execute(DATABASE_NAME, query)
    return dataframe_from_result_table(response.primary_results[0])

## 🏁 Latest Race Session

In [ ]:
# Get latest session
latest_session_query = """
GameEvents
| summarize LastEvent = max(Timestamp) by GameSessionId
| top 1 by LastEvent desc
| project GameSessionId
"""
session_df = run_query(latest_session_query)
SESSION_ID = session_df['GameSessionId'].iloc[0] if len(session_df) > 0 else None
print(f"📍 Active Session: {SESSION_ID}")

## 📈 Events Summary

In [ ]:
events_query = f"""
GameEvents
| where GameSessionId == "{SESSION_ID}"
| summarize Count = count() by EventType
| order by Count desc
"""
events_df = run_query(events_query)
fig = px.bar(events_df, x='EventType', y='Count', 
             title='Events by Type',
             color='EventType')
fig.show()

## 🏎️ Live Track Positions

In [ ]:
positions_query = f"""
GameEvents
| where GameSessionId == "{SESSION_ID}"
| where EventType == "Position"
| summarize arg_max(Timestamp, *) by PlayerId
| project PlayerId, PositionX, PositionY, Speed, LapNumber
"""
positions_df = run_query(positions_query)

# Create track visualization
fig = go.Figure()

# Draw track (oval)
import numpy as np
theta = np.linspace(0, 2*np.pi, 100)
track_x = 500 + 300 * np.cos(theta)
track_y = 300 + 200 * np.sin(theta)

fig.add_trace(go.Scatter(
    x=track_x, y=track_y,
    mode='lines',
    line=dict(color='gray', width=20),
    name='Track'
))

# Add cars
colors = {'Player1': 'red', 'Player2': 'blue', 'Player3': 'green', 'Player4': 'yellow'}
for _, row in positions_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['PositionX']], y=[row['PositionY']],
        mode='markers+text',
        marker=dict(size=20, color=colors.get(row['PlayerId'], 'white')),
        text=f"L{row['LapNumber']}",
        name=row['PlayerId']
    ))

fig.update_layout(
    title='Live Track Positions',
    xaxis=dict(range=[100, 900], visible=False),
    yaxis=dict(range=[0, 600], visible=False),
    showlegend=True
)
fig.show()

## ⏱️ Lap Times Comparison

In [ ]:
lap_times_query = f"""
GameEvents
| where GameSessionId == "{SESSION_ID}"
| where EventType == "LapComplete" or EventType == "RaceStart"
| order by Timestamp asc
| partition by PlayerId (
    project PlayerId, LapNumber, Timestamp
    | extend PrevTimestamp = prev(Timestamp)
    | where isnotnull(PrevTimestamp)
    | extend LapTimeSeconds = datetime_diff('millisecond', Timestamp, PrevTimestamp) / 1000.0
)
| project PlayerId, LapNumber, LapTimeSeconds
"""
lap_df = run_query(lap_times_query)

if len(lap_df) > 0:
    fig = px.bar(lap_df, x='LapNumber', y='LapTimeSeconds', 
                 color='PlayerId', barmode='group',
                 title='Lap Times by Player')
    fig.show()
else:
    print("No lap data yet")

## 📊 Speed Over Time

In [ ]:
speed_query = f"""
GameEvents
| where GameSessionId == "{SESSION_ID}"
| where EventType == "Position"
| summarize AvgSpeed = avg(Speed) by bin(Timestamp, 1s), PlayerId
| order by Timestamp asc
"""
speed_df = run_query(speed_query)

fig = px.line(speed_df, x='Timestamp', y='AvgSpeed', 
              color='PlayerId',
              title='Average Speed Over Time')
fig.show()

## 🔥 Track Section Heatmap

In [ ]:
heatmap_query = f"""
GameEvents
| where GameSessionId == "{SESSION_ID}"
| where EventType == "Position"
| summarize EventCount = count(), AvgSpeed = avg(Speed) by TrackSection, PlayerId
"""
heatmap_df = run_query(heatmap_query)

# Pivot for heatmap
pivot_df = heatmap_df.pivot(index='TrackSection', columns='PlayerId', values='AvgSpeed')

fig = px.imshow(pivot_df, 
                title='Average Speed by Section and Player',
                labels=dict(x='Player', y='Track Section', color='Avg Speed'),
                color_continuous_scale='RdYlGn')
fig.show()

## 🔄 Auto-Refresh Dashboard

Run this cell to continuously refresh the dashboard during a live race:

In [ ]:
# Uncomment to enable auto-refresh (Ctrl+C to stop)
# while True:
#     clear_output(wait=True)
#     # Re-run position query and display
#     positions_df = run_query(positions_query)
#     display(positions_df)
#     time.sleep(2)